In [0]:
%sql
select * from gizmobox.bronze.v_customers

Databricks data profile. Run in Databricks to view.

In [0]:
%python
df= spark.table('gizmobox.bronze.v_customers')
dbutils.data.summarize(df)

In [0]:
select count(*) as total_record_count,
count(distinct customer_id) as distinct_customer_count
 from gizmobox.bronze.v_customers;

In [0]:
select * from (
select *,
  row_number() OVER(PARTITION BY customer_id ORDER BY created_timestamp DESC) as rn 
  FROM gizmobox.bronze.v_customers
  WHERE customer_id IS NOT NULL) where rn =1;

In [0]:
CREATE OR REPLACE TABLE gizmobox.silver.customers
AS
WITH remove_duplicates(
  select * from (
select *,
  row_number() OVER(PARTITION BY customer_id ORDER BY created_timestamp DESC) as rn 
  FROM gizmobox.bronze.v_customers
  WHERE customer_id IS NOT NULL) where rn =1
)
select 
CAST(created_timestamp AS timestamp) AS created_timestamp,
customer_id,
customer_name,
date_of_birth:: DATE AS date_of_birth,
email,
member_since:: DATE AS member_since,
telephone
FROM remove_duplicates;

In [0]:
select * from gizmobox.silver.customers;